In [21]:
import pandas as pd
import os
from itertools import permutations, combinations
import re
import numpy as np

## Clorofila de las boyas

Funciones para hacer la media de un cierto rango de profundidades y para crear un único dataframe con todas las boyas, sus localizaciones y la clorofila a partir del diccionario de dataframes.

In [68]:
def add_average_column(df_dict, depth):
    for name, df in df_dict.items():
        df = df.copy()  
        
        columns = ['0.0', '0.5', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']
        
        # Select columns to average (excluding '0.5' and '1.0')
        if depth == ">1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['0.0', '0.5', '1.0']]
        if depth == "<1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['1.5','2.0','2.5', '3.0', '3.5', '4.0', '4.5', '5.0']]
        if depth == "<2":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['2.5', '3.0', '3.5', '4.0', '4.5', '5.0']]
        if depth == "=1":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['1.0']]
        if depth == "=0":
            cols_to_avg = [col for col in columns if col in df.columns and col in ['0.0', '0.5']]
        if depth == ">3":
            cols_to_avg = [col for col in columns if col in df.columns and col not in ['0.0', '0.5', '1.0', '1.5','2.0','2.5', '3.0']]

        # Compute the mean and add a new column
        if cols_to_avg:
            df['Average'] = df[cols_to_avg].mean(axis=1).round(3)
    
        # Update the dictionary with the modified DataFrame
        df_dict[name] = df

    return df_dict


def create_combined_dataframe(df_dict):
    data = []
    
    for name in sorted(df_dict.keys()):
        #if name.startswith("CTD5") or name.startswith("CTD9") or name.startswith("CTD-E5") or name.startswith("CTD-E9"):
        if name.startswith("CTD5") or name.startswith("CTD-E5"):
            continue  
        
        buoy_name = name.split('_')[0]  # Extract buoy name (CTD1, CTD2, etc.)
        df = df_dict[name]
        
        if 'Average' in df.columns:
            for _, row in df.iterrows():
                data.append({'Date': row['Date'], 'Buoy': buoy_name, 'Chl': row['Average']})
    
    combined_df = pd.DataFrame(data).sort_values(by=["Date", "Buoy"])
    combined_df["Buoy"] = combined_df["Buoy"].str.replace(r"CTD-E", "CTD", regex=True)
    return combined_df

#### UPCT

In [71]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [73]:
path = "boyaUPCT/extractedData/"
buoy_ids = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
dataframes_boyas = {}
for buoy_id in buoy_ids:
    archivos_csv = [f for f in os.listdir(path + buoy_id) if f.endswith('.csv')]
    for archivo in archivos_csv:     
        nombre_variable = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(path, buoy_id, archivo)
        if nombre_variable in ["Clorofila"]: #["Turbidez"]
            dataframes_boyas[f"{buoy_id}_{nombre_variable}"] = pd.read_csv(ruta_completa)
for key, df in dataframes_boyas.items():
    df['Date'] = pd.to_datetime(df['Date'])

In [75]:
depth_names = {"=0": "eq_0", "=1" : "eq_1", "<1" : "lt_1", "<2" : "lt_2", ">1": "gt_1", ">3": "gt_3"}
for depth, name in depth_names.items():
    dataframes_boyas = add_average_column(dataframes_boyas, depth)
    df_boyas = create_combined_dataframe(dataframes_boyas)
    df_boyas.to_csv(f"saved_files/df_boyas_upct_depth_{name}.csv", index=False)

In [76]:
df_boyas.head(2)

,Date,Buoy,Chl
1119,2017-05-19,CTD1,0.883
0,2017-05-19,CTD10,0.955


In [17]:
df_boyas.to_csv(f"saved_files/df_boyas_upct_depth_{depth_names[depth]}.csv", index=False)

#### IMIDA

In [77]:

# Ruta a la carpeta que contiene todas las boyas
path = "boyasProf/"
# Listamos los nombres de todas las boyas
buoy_ids = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
# Diccionario para guardar dataframes
dataframes_boyas_imida = {}
# Para cada boya cargamos todos sus csvs y los guardamos con key f"{bouy_id}_{variable}"
for buoy_id in buoy_ids:
    archivos_csv = [f for f in os.listdir(path + buoy_id) if f.endswith('.csv')]
    for archivo in archivos_csv:     
        nombre_variable = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(path, buoy_id, archivo)
        if nombre_variable in ["Clorofila"]:
            dataframes_boyas_imida[f"{buoy_id}_{nombre_variable}"] = pd.read_csv(ruta_completa)
for key, df in dataframes_boyas_imida.items():
    df['Date'] = pd.to_datetime(df['fecha'])

In [78]:
for key, df in dataframes_boyas_imida.items():
    nuevo_nombre_columnas = {}
    for col in df.columns:
        match = re.search(r'profundidad_(-?\d+\.?\d*)', col)
        if match:
            nuevo_nombre_columnas[col] = str(abs(float(match.group(1))))
    dataframes_boyas_imida[key].rename(columns=nuevo_nombre_columnas, inplace=True)

# Lista de profundidades deseadas como strings
cols_to_keep = ['Date', '0.0', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0']

for key, df in dataframes_boyas_imida.items():
    columnas_filtradas = [col for col in df.columns if col in cols_to_keep]
    dataframes_boyas_imida[key] = df[columnas_filtradas]

In [ ]:
# VER COMO AÑADIR LOS DATOS DE LA IMAGEN NUEVA DESDE AQUÍ

In [80]:
#depth = "=0" # >1, <1, <2, =0, =1
depth_names = {"=0": "eq_0", "=1" : "eq_1", "<1" : "lt_1", "<2" : "lt_2", ">1": "gt_1", ">3": "gt_3"}
for depth, name in depth_names.items():
    dataframes_boyas_imida = add_average_column(dataframes_boyas_imida, depth)
    df_boyas = create_combined_dataframe(dataframes_boyas_imida)
    df_boyas.to_csv(f"saved_files/df_boyas_imida_depth_{name}.csv", index=False)

In [43]:
df_boyas.to_csv(f"saved_files/df_boyas_imida_depth_{depth_names[depth]}.csv", index=False)

#### UPCT e IMIDA

In [82]:
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_") and archivo.endswith(".csv") and "merge" not in archivo:
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_boyas[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [83]:
# Resultado: dict con DataFrames combinados por sufijo
dfs_combinados = {}

# Agrupar por sufijo
grupos_por_sufijo = {}

for nombre, df in dfs_boyas.items():
    # Extraer sufijo (lo que va después de 'depth_')
    match = re.search(r'depth_(\w+_\d+)', nombre)
    if match:
        sufijo = match.group(1)
        grupos_por_sufijo.setdefault(sufijo, []).append(df)

# Función para combinar dos DataFrames según la lógica que mencionaste
def combinar_dataframes(df1, df2):
    merged = pd.merge(df1, df2, on=['Date', 'Buoy'], how='outer', suffixes=('_1', '_2'))
    
    def combinar_chl(row):
        chl1 = row['Chl_1']
        chl2 = row['Chl_2']
        if pd.notna(chl1) and pd.notna(chl2):
            return (chl1 + chl2) / 2
        elif pd.notna(chl1):
            return chl1
        elif pd.notna(chl2):
            return chl2
        else:
            return np.nan

    merged['Chl'] = merged.apply(combinar_chl, axis=1)
    return merged[['Date', 'Buoy', 'Chl']]

# Combinar los pares
for sufijo, lista_dfs in grupos_por_sufijo.items():
    if len(lista_dfs) == 2:
        df1, df2 = lista_dfs
        combinado = combinar_dataframes(df1, df2)
        dfs_combinados[sufijo] = combinado
    else:
        print(f"Sufijo '{sufijo}' no tiene exactamente dos DataFrames asociados.")



In [84]:
for df_name, df in dfs_combinados.items():
    df.to_csv(f"saved_files/df_boyas_merge_depth_{df_name}.csv", index=False)

## Reflectancias

In [86]:
import rasterio
import numpy as np
from datetime import datetime

In [87]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [88]:
def extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping, net_set):

    """
    Extract pixel values from TIFF files for each buoy in loc_boyas and for each specified date.
    """
    results = []
    target_dates = sorted(set(target_dates))
    if net_set == "C2X-Complex":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XComplexNets' in f]
    elif net_set == "C2X":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XNets' in f]
    elif net_set == "C2RCC":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2RCC' in f]
    else:
        print("Net Set no reconocido")

    for date in target_dates:
        date_to_find = date.replace('-', '')
        matching = [f for f in tif_files if date_to_find in f]
        if matching:
            tiff_file = os.path.join(folder_path, matching[0])  

            with rasterio.open(tiff_file) as dataset:
                print(f"Processing {tiff_file}")
                bands = dataset.read()

                for _, row in loc_boyas.iterrows():
                    buoy_id = row["CodPuntoControl"].replace('-', '').strip()
                    lat = int(round(row["LatitudEPSG32630"]))
                    lon = int(round(row["LongitudEPSG32630"]))

                    try:
                        row_idx, col_idx = dataset.index(lon, lat)
                        #print(row_idx, col_idx)

                        if grouping == "3x3":
                            offset = 1
                        elif grouping == "5x5":
                            offset = 2
                        elif grouping == "9x9":
                            offset = 4
                        else:
                            offset = 0

                        if offset > 0:
                            window = (
                                slice(max(row_idx - offset, 0), min(row_idx + offset + 1, dataset.height)),
                                slice(max(col_idx - offset, 0), min(col_idx + offset + 1, dataset.width))
                            )
                            reflectances = bands[:, window[0], window[1]]
                            values = np.median(reflectances, axis=(1, 2))
                        else:
                            values = bands[:, row_idx, col_idx]

                        results.append({
                            "Date": date,
                            "Buoy": buoy_id,
                            "Latitude": lat,
                            "Longitude": lon,
                            **{f"Band_{i+1}": val for i, val in enumerate(values)}
                        })

                    except IndexError:
                        print(f"Skipping {buoy_id} on {date}: Coordinates out of raster bounds")

    return pd.DataFrame(results)

In [90]:

folder_path = "Copernicus/SAFE_downloads/processed/"
target_dates = [
    '2016-08-09', '2016-09-08', '2017-06-30', '2018-02-20', '2018-03-07', 
    '2018-05-16', '2018-06-20', '2018-07-10', '2018-08-29', '2018-10-03',
    '2018-11-07', '2019-03-12', '2019-06-25', '2019-07-10', '2019-07-30', 
    '2019-08-14', '2019-09-18', '2019-09-28', '2019-10-03', '2019-11-27', '2020-02-20', 
    '2020-03-11', '2020-12-21', '2021-01-05', '2021-04-20', '2021-05-20',
    '2021-06-14', '2021-08-13', '2021-11-11', '2021-11-26', '2021-12-01', 
    '2022-02-24', '2022-07-14', '2022-07-29', '2022-08-03', '2022-09-07', 
    '2022-09-27', '2023-01-10', '2023-03-01', '2023-03-16', '2023-04-20', 
    '2023-05-25'
]

groupings = ["1x1", "3x3", "5x5", "9x9"]
net_set = ["C2X", "C2X-Complex", "C2RCC"]
for grouping in groupings:
    for net in net_set:
        df_tiffs = extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping, net)
        df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])
        df_tiffs.to_csv(f"saved_files/df_tifs_{net}_{grouping}.csv", index=False)


Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20160809T105032_N0500_R051_T30SXG_20231030T221951_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20160908T105022_N0500_R051_T30SXG_20231030T134748_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2B_MSIL1C_20170630T105029_N0500_R051_T30SXG_20231017T002243_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20180220T105051_N0500_R051_T30SXG_20230910T163638_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2B_MSIL1C_20180307T105019_N0500_R051_T30SXG_20230908T224638_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2B_MSIL1C_20180516T105029_N0500_R051_T30SXG_20230902T110634_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20180620T105031_N0500_R051_T30SXG_20230811T201235_C2XNets_10m.tif
Processing Copernicus/SAFE_downloads/processed/S2A_MSIL1C_20180710T105031_N0500_R051_T30SXG_20230817T062327_C2XNets_10m.tif
Processi

## Unión de clorofila y reflectancias

In [104]:
# Cargamos los csv de los tifs
path = "saved_files/"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tifs_") and archivo.endswith(".csv") and "planet" not in archivo:
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_tifs[nombre_sin_extension] = pd.read_csv(ruta_completa)

# Y de las boyas
        # COGEMOS SOLO EL MERGE DE IMIDA Y UPCT
path = "saved_files/"
dfs_boyas = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_boyas_merge") and archivo.endswith(".csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_boyas[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [29]:
set(list(dfs_boyas["df_boyas_imida_depth_eq_0"]["Date"].unique()) + list(dfs_boyas["df_boyas_upct_depth_eq_0"]["Date"].unique()))

{'2016-06-08',
 '2016-06-29',
 '2016-08-02',
 '2016-08-04',
 '2016-08-09',
 '2016-08-22',
 '2016-08-25',
 '2016-08-26',
 '2016-08-27',
 '2016-08-30',
 '2016-08-31',
 '2016-09-01',
 '2016-09-06',
 '2016-09-08',
 '2016-09-22',
 '2016-09-29',
 '2016-10-06',
 '2016-10-14',
 '2016-10-21',
 '2016-10-28',
 '2016-11-03',
 '2016-11-11',
 '2016-11-18',
 '2016-11-22',
 '2016-12-02',
 '2016-12-13',
 '2016-12-21',
 '2016-12-29',
 '2017-01-12',
 '2017-01-26',
 '2017-02-03',
 '2017-02-10',
 '2017-02-16',
 '2017-02-24',
 '2017-03-02',
 '2017-03-22',
 '2017-03-30',
 '2017-04-05',
 '2017-04-12',
 '2017-04-18',
 '2017-04-25',
 '2017-05-04',
 '2017-05-10',
 '2017-05-19',
 '2017-05-24',
 '2017-06-02',
 '2017-06-06',
 '2017-06-13',
 '2017-06-21',
 '2017-06-30',
 '2017-07-05',
 '2017-07-12',
 '2017-07-19',
 '2017-07-26',
 '2017-08-08',
 '2017-08-16',
 '2017-08-23',
 '2017-08-30',
 '2017-09-06',
 '2017-09-11',
 '2017-09-20',
 '2017-09-27',
 '2017-10-04',
 '2017-10-11',
 '2017-10-17',
 '2017-10-25',
 '2017-11-

In [105]:
band_names = {
    "Band_1": "rhow_B1",
    "Band_2": "rhow_B2",
    "Band_3": "rhow_B3",
    "Band_4": "rhow_B4",
    "Band_5": "rhow_B5",
    "Band_6": "rhow_B6",
    "Band_7": "rhow_B7",
    "Band_8": "rhow_B8",
    "Band_9": "rhown_B1",
    "Band_10": "rhown_B2",
    "Band_11": "rhown_B3",
    "Band_12": "rhown_B4",
    "Band_13": "rhown_B5",
    "Band_14": "rhown_B6",
}
for nombre_df, df in dfs_tifs.items():
    dfs_tifs[nombre_df] = df.rename(columns=band_names)

In [99]:
dfs_tifs["df_tifs_c2x-complex-nets_1x1"].head(3)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,rhow_B7,rhow_B8,rhown_B1,rhown_B2,rhown_B3,rhown_B4,rhown_B5,rhown_B6,Band_15,Band_16
0,2016-08-09,CTD1,4187246,695025,0.011593,0.016915,0.028795,0.014279,0.013503,0.003823,0.003727,0.001509,0.011479,0.016701,0.028970,0.014401,0.012486,0.003870,0.544999,-8.000000e-45
1,2016-08-09,CTD2,4181518,693105,0.011363,0.014885,0.021044,0.012019,0.009860,0.003023,0.003000,0.001253,0.011300,0.014878,0.020830,0.011859,0.009783,0.002927,0.218907,-8.000000e-45
2,2016-08-09,CTD3,4181698,695238,0.013396,0.017670,0.024609,0.014144,0.011111,0.003436,0.003327,0.001397,0.013257,0.017724,0.024332,0.013883,0.011231,0.003276,0.161813,-8.000000e-45


In [101]:
dfs_boyas["df_boyas_merge_depth_gt_1"].head(3)

,Date,Buoy,Chl
0,2017-05-19,CTD1,1.0715
1,2017-05-19,CTD10,0.8850
2,2017-05-19,CTD11,1.0495


In [106]:
merge_dict = {}

for df_tif_name, df_tif in dfs_tifs.items():
    for df_boya_name, df_boya in dfs_boyas.items():
        #print(df_tif_name[8:], df_boya_name[9:])
        merge_dict[f"{df_tif_name[8:]}_{df_boya_name[9:]}"] = df_tif.merge(df_boya, how="inner", on=["Date", "Buoy"])    

In [107]:
for df_name, df in merge_dict.items():
    df.to_csv(f"saved_files/dataset/{df_name}.csv", index=False)

## Creación de features

In [108]:
# Cargamos los csv de los tifs
path = "saved_files/dataset/"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith(".csv") and not archivo.endswith("_features.csv") :
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension] = pd.read_csv(ruta_completa)

Fórmulas a utilizar:
- Diferencia normalizada $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_1) + R(\lambda_2)}$$
- Dall-Gitelson $$\left(\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}\right) \times R(\lambda_3)$$
- Diferencia normalizada 4 bandas $$\frac{R(\lambda_1) - R(\lambda_2)}{R(\lambda_3) + R(\lambda_4)}$$
- Diferencia inversas $$\frac{1}{R(\lambda_1)}-\frac{1}{R(\lambda_2)}$$
- Diferencia relación 4 bands $$\frac{R(\lambda_1)}{R(\lambda_2)}-\frac{R(\lambda_3)}{R(\lambda_4)}$$
- Suma normalizada 3 bands $$\frac{R(\lambda_i) + R(\lambda_{i+2})}{R(\lambda_i) + R(\lambda_{i+1})}$$ donde $\lambda_{i+2} > \lambda_{i+1} > \lambda_i$

In [109]:
def diferencia_normalizada(band1, band2):
    value = (band1 - band2)/(band1 + band2)
    return value.round(3)

def dall_gitelson(band1, band2, band3):
    value = (1/(band1) - 1/(band2))*(band3)
    return value.round(3)

def diferencia_normalizada_4bandas(band1, band2, band3, band4):
    value = (band1 - band2)/(band3 + band4)
    return value.round(3)

def diferencia_inversas(band1, band2):
    value = 1/(band1) - 1/(band2)
    return value.round(3)

def diferencia_relacion_4bandas(band1, band2, band3, band4):
    value = band1/band2 - band3/band4
    return value.round(3)

def suma_normalizada_3bandas(band1, band2, band3):
    value = (band1 + band3)/(band1 + band2)
    return value.round(3)

Añadimos la diferencia normalizada y diferencia de inversas para las bandas de Ultra Blue, Blue, Green , Red, NIR1; lo hacemos dos veces, con rhow y con rhown.

In [110]:
index_list = []
def add_two_band_difs(data, bands):
    for i, band1 in enumerate(bands):
        for band2 in bands[i+1:]:
            colname_dif_norm = f"dif_norm_{band1}_{band2}"
            data[colname_dif_norm] = diferencia_normalizada(data[band1], data[band2])
            index_list.append(colname_dif_norm)
            colname_dif_inv = f"dif_inv_{band1}_{band2}"
            data[colname_dif_inv] = diferencia_inversas(data[band1], data[band2])
            index_list.append(colname_dif_inv)
    return data

Añadimos la relación de Dall-Gitelson, evitando repeticiones por la simetría de $f(b1,b2,b3)=−f(b2,b1,b3)$

In [111]:
index_dall_gitelson_list = []
def add_dall_gitelson(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita repeticiones de pares
        for band3 in bands:
            if band3 not in (band1, band2):  # evitar que band3 sea igual a los anteriores
                colname = f"dall_gitelson_{band1}_{band2}_{band3}"
                data[colname] = dall_gitelson(data[band1], data[band2], data[band3])
                index_dall_gitelson_list.append(colname)
    return data

Añadimos el índice de tipo diferencia normalizada que utiliza 4 bands, forzando a que estas 4 sean diferentes y evitando redundancia por las simetrías $(b1−b2)/(b3+b4)=(b1−b2)/(b4+b3)$ y $(b1−b2)/(b3+b4)=−(b2−b1)/(b3+b4)$

In [112]:
index_dif_norm_4bands_list = []
def add_norm_dif_4bands(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita invertir band1 y band2
        for band3, band4 in combinations(bands, 2):  # evita invertir band3 y band4
            # Asegurar que todas las bandas son distintas
            if len({band1, band2, band3, band4}) == 4:
                colname = f"dif_norm_4_bands_{band1}_{band2}_{band3}_{band4}"
                data[colname] = diferencia_normalizada_4bandas(
                    data[band1], data[band2], data[band3], data[band4]
                )
                index_dif_norm_4bands_list.append(colname)
    return data

Añadimos el cociente entre dos parejas de bandas, también evitando simetría, en este caso:

$\frac{b1}{b2} - \frac{b3}{b4} = -\left(\frac{b3}{b4} - \frac{b1}{b2} \right)$

In [113]:
index_dif_rel_4bands_list = []

def add_index_dif_rel_4bands(data, bands):
    for band1, band2, band3, band4 in permutations(bands, 4):
        # Evitar redundancias por simetría de términos
        # Criterio: solo aceptamos combinaciones donde el primer término es "menor" que el segundo
        if (band1, band2) < (band3, band4):  # evita generar la versión espejo con signo opuesto
            colname = f"dif_rel_4bands_{band1}_{band2}_{band3}_{band4}"
            data[colname] = diferencia_relacion_4bandas(
                data[band1], data[band2], data[band3], data[band4]
            )
            index_dif_rel_4bands_list.append(colname)
    return data

Para este último caso, solamente añadimos dos combinaciones (a mano), ya que al necesitar tres longitudes de onda diferentes y ordenadas de mayor a menor, y al trabajar con cuatro, solamente quedan dos opciones:
$\frac{Blue + Red}{Blue + Green}$ y $\frac{Green + NIR}{Green + Red}$

In [114]:
def add_index_sum_norm_3bands(data, bands):
    band1, band2, band3, band4 = bands

    colname = f"sum_norm_3bands_{band1}_{band3}_{band2}"
    data[colname] = suma_normalizada_3bandas(data[band1], data[band2], data[band3])

    colname = f"sum_norm_3bands_{band2}_{band4}_{band3}"
    data[colname] = suma_normalizada_3bandas(data[band2], data[band3], data[band4])
       
    return data

Aplicamos todas las fórmulas anteriores sobre los dataframes del diccionario de dataframes, para los conjuntos de bandas de rhow y rhown.

In [115]:
band_sets = [
    ['rhow_B2', 'rhow_B3', 'rhow_B4', 'rhow_B5'],
    ['rhown_B2', 'rhown_B3', 'rhown_B4', 'rhown_B5']
]
for nombre_df, df in dfs.items():
    for bands_to_use in band_sets:
        df = add_two_band_difs(df, bands_to_use)
        df = add_dall_gitelson(df, bands_to_use)
        df = add_norm_dif_4bands(df, bands_to_use)
        df = add_index_dif_rel_4bands(df, bands_to_use)
        df = add_index_sum_norm_3bands(df, bands_to_use)
    dfs[nombre_df] = df 

Ya tenemos todos los dataframes con todas las columnas.

In [117]:
dfs["C2X_3x3_merge_depth_lt_1"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B2_rhown_B5_rhown_B3_rhown_B4,dif_rel_4bands_rhown_B2_rhown_B5_rhown_B4_rhown_B3,dif_rel_4bands_rhown_B3_rhown_B2_rhown_B4_rhown_B5,dif_rel_4bands_rhown_B3_rhown_B2_rhown_B5_rhown_B4,dif_rel_4bands_rhown_B3_rhown_B4_rhown_B5_rhown_B2,dif_rel_4bands_rhown_B3_rhown_B5_rhown_B4_rhown_B2,dif_rel_4bands_rhown_B4_rhown_B2_rhown_B5_rhown_B3,dif_rel_4bands_rhown_B4_rhown_B3_rhown_B5_rhown_B2,sum_norm_3bands_rhown_B2_rhown_B4_rhown_B3,sum_norm_3bands_rhown_B3_rhown_B5_rhown_B4
0,2016-08-09,CTD1,4187246,695025,0.005731,0.014207,0.038569,0.018639,0.017016,0.004788,...,-1.158,0.414,1.522,1.862,0.947,1.121,0.905,-0.625,0.625,0.949
1,2016-08-09,CTD2,4181518,693105,0.006148,0.011972,0.023885,0.011034,0.010585,0.003231,...,-0.969,0.696,0.931,1.157,1.275,1.427,0.542,-0.390,0.643,0.966


Renombramos columnas para que los nombres no incluyan varias veces rhow / rhown. Es decir, que index_diff_ratio_rhow_B1_rhow_B4_rhow_B2_rhow_B5 sea solamente index_diff_ratio_rhow_B1_B4_B2_B5.

In [118]:
import re

def compactar_prefijos_columnas(df):
    nuevo_nombre_columnas = {}

    for col in df.columns:
        # Detectar columnas con patrones tipo index_algo_rhow_B1_rhow_B2_...
        if re.search(r'(rhow|rhown)(_B\d+)+', col):
            partes = col.split('_')
            base = []
            bandas = []
            prefijo = None

            for parte in partes:
                if parte in ['rhow', 'rhown']:
                    if not prefijo:
                        prefijo = parte
                elif parte.startswith('B'):
                    bandas.append(parte)
                else:
                    base.append(parte)

            if prefijo and bandas:
                #nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                if base:
                    nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                else:
                    nuevo_nombre = f"{prefijo}_{'_'.join(bandas)}"

                nuevo_nombre_columnas[col] = nuevo_nombre

    # Renombrar columnas
    df = df.rename(columns=nuevo_nombre_columnas)
    return df


In [119]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = compactar_prefijos_columnas(df)

In [120]:
dfs["C2X_3x3_merge_depth_lt_1"].head(2)

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhown_B2_B5_B3_B4,dif_rel_4bands_rhown_B2_B5_B4_B3,dif_rel_4bands_rhown_B3_B2_B4_B5,dif_rel_4bands_rhown_B3_B2_B5_B4,dif_rel_4bands_rhown_B3_B4_B5_B2,dif_rel_4bands_rhown_B3_B5_B4_B2,dif_rel_4bands_rhown_B4_B2_B5_B3,dif_rel_4bands_rhown_B4_B3_B5_B2,sum_norm_3bands_rhown_B2_B4_B3,sum_norm_3bands_rhown_B3_B5_B4
0,2016-08-09,CTD1,4187246,695025,0.005731,0.014207,0.038569,0.018639,0.017016,0.004788,...,-1.158,0.414,1.522,1.862,0.947,1.121,0.905,-0.625,0.625,0.949
1,2016-08-09,CTD2,4181518,693105,0.006148,0.011972,0.023885,0.011034,0.010585,0.003231,...,-0.969,0.696,0.931,1.157,1.275,1.427,0.542,-0.390,0.643,0.966


Guardamos los dataframes como csvs, con el mismo nombre que tenían pero añadiendo "_features" al final.

In [121]:
for df_name, df in dfs.items():
    df.to_csv(f"saved_files/dataset/{df_name}_features.csv", index=False)